# 🚀 EmbedIA: Redes Neuronales y Machine Learning en Microcontroladores

¡Imagina ejecutar un modelo de reconocimiento de dígitos (como MNIST) en un Arduino o ESP32 pequeño, sin bibliotecas pesadas ni conectividad en la nube—EmbedIA lo hace posible! Este framework convierte modelos de TensorFlow y Scikit-learn en código C optimizado para microcontroladores de bajo consumo.

En este tutorial, entrenaremos un clasificador simple de dígitos manuscritos y lo exportaremos a C usando EmbedIA. ¡Al final, tendrás un proyecto listo para compilar y probar en hardware real!

## 🛠️ Instalando EmbedIA desde GitHub (solo Colab)

Si estás usando Google Colab, clona el repositorio para acceder al framework.

In [1]:
import os

%cd /content/

if not os.path.exists("EmbedIA"):
    !git clone https://github.com/Embed-ML/EmbedIA.git

%cd EmbedIA

## 📂 Cargando el Dataset

Usaremos el dataset de dígitos de Scikit-learn (similar a MNIST pero más ligero). Aplicaremos normalización para mejorar el rendimiento en microcontroladores.

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from sklearn.model_selection import train_test_split

IMG_SHAPE = (14, 14)

def load_dataset(
        image_size=IMG_SHAPE,
        train_samples=15000,
        test_samples=3000):

    # Cargar MNIST
    (X_train, y_train), (X_test, y_test) = mnist.load_data()

    print('X_train original:', X_train.shape)
    print('X_test original :', X_test.shape)

    # Normalizar
    X_train = X_train.astype("float32") / 255.0
    X_test  = X_test.astype("float32") / 255.0

    # Agregar dimensión de canal
    X_train = np.expand_dims(X_train, axis=-1)
    X_test  = np.expand_dims(X_test, axis=-1)

    # Redimensionar
    X_train = tf.image.resize(X_train, image_size).numpy()
    X_test  = tf.image.resize(X_test, image_size).numpy()

    # Reducir tamaño del dataset
    X_train = X_train[:train_samples]
    y_train = y_train[:train_samples]

    X_test = X_test[:test_samples]
    y_test = y_test[:test_samples]

    print('X_train redimensionado:', X_train.shape)
    print('X_test redimensionado :', X_test.shape)

    return X_train, X_test, y_train, y_test

x_train, x_test, y_train, y_test = load_dataset()

print('x_train.shape', x_train.shape)
print(' x_test.shape', x_test.shape)

## 🖼️ Visualizando Muestras

Mostremos algunos dígitos del dataset para explorar los datos.

In [3]:
import matplotlib.pyplot as plt
import numpy as np

img_x_digit = 6

fig, axes = plt.subplots(img_x_digit, 10, figsize=(10, 6))  # Cuadrícula ajustada para proporción adecuada

for digit in range(10):
    digit_indices = np.where(y_train == digit)[0]  # Encontrar índices de imágenes con el dígito actual

    for row in range(img_x_digit):
        idx = digit_indices[row] if row < len(digit_indices) else digit_indices[0]
        axes[row, digit].imshow(x_train[idx].reshape(IMG_SHAPE), cmap='gray_r')
        axes[row, digit].set_title(f'{digit}', fontsize=10)
        axes[row, digit].axis('off')

plt.suptitle(f'{img_x_digit*10} Ejemplos de Dígitos ({img_x_digit} por Clase)', fontsize=16, y=1.05)
plt.tight_layout()
plt.show()

## 🤖 Creación del Modelo

Construiremos una red neuronal convolucional (CNN) simple adecuada para microcontroladores: ligera y eficiente.

In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    RandomRotation,
    RandomTranslation,
    RandomZoom
)

def create_model(x_train, y_train, y_test):

    classes = int(max(y_test.max(), y_train.max()) + 1)

    model = Sequential(name="EmbedIA_model")

    model.add(Input(shape=x_train[0].shape))

    # Aumento de datos
    model.add(RandomRotation(0.08))
    model.add(RandomTranslation(0.08, 0.08))
    model.add(RandomZoom(0.12))

    # CNN
    model.add(Conv2D(16, (3,3), padding='same', activation='relu'))
    model.add(MaxPooling2D((2,2)))

    model.add(Conv2D(32, (3,3), padding='same', activation='relu'))
    model.add(MaxPooling2D((2,2)))

    model.add(Flatten())

    model.add(Dense(16, activation='relu'))
    model.add(Dense(classes, activation='softmax'))

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['acc']
    )

    return model

model = create_model(x_train, y_train, y_test)
model.summary()

## 🏋️ Entrenamiento del Modelo

Entrenaremos durante varias épocas. Monitorea la precisión de validación.

In [5]:
from tensorflow.keras.callbacks import EarlyStopping
epochs = 50
batch_size = 64

# Entrenar el modelo
history = model.fit(
    x_train,
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(x_test, y_test),
    callbacks=[EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True)]
)

## 📈 Evaluación y Visualización

Graficaremos la precisión de entrenamiento y validación. ¡Observa qué tan rápido converge!

In [6]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

# Historial de precisión
df = pd.DataFrame({
    'Épocas': range(len(history.history['acc'])),
    'Entrenamiento': history.history['acc'],
    'Validación': history.history['val_acc']
})

# Matriz de confusión
y_pred = np.argmax(model.predict(x_test), axis=1)

cm = confusion_matrix(y_test, y_pred)

# nombres de clases
class_names = [str(i) for i in range(cm.shape[0])]

# Subplots
fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.65, 0.35],
    subplot_titles=(
        f'📈 Precisión del Modelo (Validación: {history.history["val_acc"][-1]:.2%})',
        '🧩 Matriz de Confusión'
    )
)

# Curvas de precisión
fig.add_trace(
    go.Scatter(
        x=df['Épocas'],
        y=df['Entrenamiento'],
        mode='lines',
        name='Entrenamiento'
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=df['Épocas'],
        y=df['Validación'],
        mode='lines',
        name='Validación'
    ),
    row=1,
    col=1
)

# Heatmap de matriz de confusión
fig.add_trace(
    go.Heatmap(
        z=cm,
        x=class_names,
        y=class_names,
        text=cm,
        texttemplate="%{text}",
        textfont={"size": 14},
        colorscale='Blues',
        showscale=True
    ),
    row=1,
    col=2
)

# Layout
fig.update_layout(
    template='plotly_white',
    width=1200,
    height=450,

    margin=dict(t=80),

    legend=dict(
        orientation="h",
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5
    )
)

# Ejes de precisión
fig.update_xaxes(title_text='Épocas', row=1, col=1)
fig.update_yaxes(title_text='Precisión', row=1, col=1)

# Ejes de matriz de confusión
fig.update_xaxes(title_text='Predicho', row=1, col=2)
fig.update_yaxes(title_text='Real', row=1, col=2)

fig.show()

## 📤 Donde ocurre la magia: conversión a código C

Transformando el modelo en código C optimizado usando EmbedIA. Configura los ajustes de exportación y genera el proyecto objetivo.

In [7]:
#@title ⚙️ Configurar y Ejecutar Exportador { display-mode: "form" }

import sys
# Agregar carpeta padre al path para ubicar EmbedIA
sys.path.insert(0, '..')

import joblib
from tensorflow.keras.models import load_model
from embedia.project_generator import ProjectGenerator
from embedia.model_generator.project_options import (
    ModelDataType,
    DebugMode,
    ProjectFiles,
    ProjectOptions,
    ProjectType
)

#@markdown ---
#@markdown ### Configuración de Exportación

OUTPUT_FOLDER = 'outputs/' #@param {type:"string"}
PROJECT_NAME = 'mnist_project' #@param {type:"string"}

options = ProjectOptions()

# Establecer ubicación de la carpeta EmbedIA
embedia_folder = '../EmbedIA/embedia'
options.embedia_folder = embedia_folder

#@markdown ---
#@markdown ### Configuración del Proyecto

# Tipo de proyecto
project_type_dict = {
    "ARDUINO": ProjectType.ARDUINO,
    "C": ProjectType.C,
    "CODEBLOCK": ProjectType.CODEBLOCK,
    "CPP": ProjectType.CPP,
    "CMAKE_C": ProjectType.CMAKE_C,
    "CMAKE_CPP": ProjectType.CMAKE_CPP,
}
project_type = "C" #@param ["ARDUINO", "C", "CODEBLOCK", "CPP", "CMAKE_C", "CMAKE_CPP"]
options.project_type = project_type_dict[project_type]

# Tipo de dato
data_type_dict = {
    "FLOAT": ModelDataType.FLOAT,
    "FIXED32": ModelDataType.FIXED32,
    "FIXED16": ModelDataType.FIXED16,
    "FIXED8": ModelDataType.FIXED8,
    "QUANT8": ModelDataType.QUANT8,
    "BINARY": ModelDataType.BINARY,
}
data_type = "FIXED16" #@param ["FLOAT", "FIXED32", "FIXED16", "FIXED8", "QUANT8", "BINARY"]
options.data_type = data_type_dict[data_type]

# Modo de depuración
debug_mode_dict = {
    "DISCARD": DebugMode.DISCARD,
    "DISABLED": DebugMode.DISABLED,
    "HEADERS": DebugMode.HEADERS,
    "DATA": DebugMode.DATA,
}
debug_mode = "DISABLED" #@param ["DISCARD", "DISABLED", "HEADERS", "DATA"]
options.debug_mode = debug_mode_dict[debug_mode]

# Archivos a exportar
files_export_dict = {
    "ALL": ProjectFiles.ALL,
    "MAIN": {ProjectFiles.MAIN},
    "MODEL": {ProjectFiles.MODEL},
    "LIBRARY": {ProjectFiles.LIBRARY},
}
files_export = "ALL" #@param ["ALL", "MAIN", "MODEL", "LIBRARY"]
options.files = files_export_dict[files_export]

# Datos de ejemplo para exportación
samples = x_test[0:15]
ids = y_test[0:15]

options.example_data = samples
options.example_ids = ids

#@markdown Marque para limpiar la carpeta de salida antes de exportar:
clean_output = True #@param {type:"boolean"}
options.clean_output = clean_output
#@markdown ---

############# Generar Proyecto #############

generator = ProjectGenerator(options)
generator.create_project(OUTPUT_FOLDER, PROJECT_NAME, model, options)

print("Proyecto", PROJECT_NAME, "exportado a", OUTPUT_FOLDER,'\n\n')

## ⬇️ Descargar Proyecto Generado (solo Colab)

¡Todo listo! Descarga el ZIP y compílalo en tu IDE favorito. Pruébalo en un microcontrolador y comparte tus resultados en el repositorio.

In [8]:
from google.colab import files

!zip -r embedia_project.zip 'outputs/mnist_project'
files.download('/content/EmbedIA/embedia_project.zip')

## 🛠️ Compilando el Proyecto C Generado

Esta sección compila el proyecto C generado por EmbedIA. Navega al directorio del proyecto, crea una carpeta de compilación y luego compila.
Para este ejemplo solo es posible compilar para los tipos de proyectos `C`,`CPP`,`CMAKE_C`,`CMAKE_CPP` y `CODEBLOCK`.

In [9]:
from pathlib import Path
import os
import subprocess

project = Path("/content/EmbedIA/outputs/mnist_project")
build = project / "build"
binary = build / "mnist_project"


if project_type == 'ARDUINO':
    print('No se pueden compilar archivos de Arduino')
else:
    original_cwd = Path.cwd()

    build.mkdir(parents=True, exist_ok=True)

    if project_type in ["CMAKE_C", "CMAKE_CPP"]:
        commands = [
            ["cmake", ".."],
            ["make"]
        ]
    else:
        sources = [
            *project.rglob("*.c"),
            *project.rglob("*.cpp")
        ]
        assert sources, "No se encontraron archivos fuente"

        compiler = "g++" if any(f.suffix == ".cpp" for f in sources) else "gcc"

        commands = [[ compiler, *map(str, sources), "-o", str(binary) ]]

    commands.append([str(binary)])

    for cmd in commands:
        print(">", " ".join(cmd))

        result = subprocess.run(cmd, cwd=build)

        if result.returncode != 0:
            raise RuntimeError(f"Compilación fallida: {' '.join(cmd)}")

    os.chdir(original_cwd)

## ▶️ Ejecutando el Proyecto Compilado

Esta sección ejecuta el binario compilado del proyecto C generado. Después de la ejecución, el directorio de trabajo original se restaura para asegurar que las celdas posteriores se ejecuten en el entorno esperado.

In [10]:
assert binary.exists(), f"Binario no encontrado: {binary}"

result = subprocess.run(
    [str(binary)],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout)

## Área de Pruebas

In [19]:
import numpy as np
#from PIL import Image, ImageOps
from web_utils.draw_panel import DrawPanel

IMG_SHAPE=model.input_shape[1:-1]

dp = DrawPanel()
image = dp.draw(size=IMG_SHAPE, line_width=1.2, scale = 12)

# obtiene canal de dibujo y normaliza a [0, 1]
gs_image = np.array(image)[:,:,3]/255

# prepara formato para funcion de prediccion
gs_image = gs_image.reshape(1,*IMG_SHAPE,1)

# obtiene salida (sofmax)
resp = model.predict(gs_image, verbose=0)

# posición de la neurona de salida con mayor valor
digito = np.argmax(resp)

print("\033[1mEl trazo dibujado corresponde al dígito %d\033[0m" % digito)